# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abc085455-byte/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring** (Lane 2 of the four predefined lanes).

I'm picking this lane because it maps onto a real, recurring decision — which content page an editor should look at first — and the starter dataset already carries the exact signals (impressions, CTR, position, trend, freshness) that this lane's baseline rules use. The output is a ranked queue with reason codes, not a vague "insight," and the starter pipeline's own results (baseline ROC-AUC 0.627 vs. random forest 0.750, precision@50 going from 0.240 to 0.740) show a learned ranking clearly beats the simple rule on this data — a good sign this lane is worth the next 7 weeks. I can confirm or swap this by end of Week 4 if the data pushes me elsewhere.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Out of thousands of content pages, which ones should a content/SEO editor review first this week, given they only have time for a small number?

**Who acts, and how:** A content editor (or their manager, prioritizing the team's queue) uses my ranked list to pick the top N pages and choose an action per page — refresh the copy, fix the title/meta, or leave it and just monitor.

**Cost of a wrong call:**
- *False positive* — I rank a page high but it wasn't really a priority: the editor loses 1–3 hours rewriting a page that didn't need it. Wasted labor, no gain.
- *False negative* — a real declining, high-demand page never surfaces in my top ranks: it keeps losing visibility and clicks for weeks longer before anyone notices, which compounds and is the more expensive mistake.

Because a missed decline is costlier than one wasted review hour, my ranking should lean toward catching real decliners near the top of the queue, not just chasing raw precision.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Loading `data/raw/content_refresh_anonymized.csv` (30,000 rows, 32 clients) and applying the lane guide's own eligibility filter (`impressions_90d > 0`, `content_age_days >= 90`) keeps all 30,000 rows. Three numbers from that eligible set (computed in the code cell below):

- **43.8%** of pages (13,152 / 30,000) match the guide's `declining_with_demand` reason code (`trend_direction == "down"` and `impressions_90d >= 100`) — a large, non-trivial candidate pool, not a handful of edge cases.
- **32.5%** of pages (9,759 / 30,000) match `low_ctr_visible_page` (decent position, ≥500 impressions, CTR < 0.5%) — a second, largely independent opportunity signal worth ranking on its own.
- **Median impressions_90d is 731** across eligible pages, so most candidates already carry real search demand — this lane isn't built on noise.

For comparison, the AI-referral freestyle direction only has AI-session data on about 0.1% of daily rows in the full warehouse — this lane's population is far larger and easier to work with for a 7-week project.


In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("rows:", len(df), "| clients:", df["client_id"].nunique())

eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
print("eligible rows:", len(eligible))

declining_with_demand = (eligible["trend_direction"] == "down") & (eligible["impressions_90d"] >= 100)
low_ctr_visible_page = (
    (eligible["impressions_90d"] >= 500)
    & (eligible["avg_position"] > 0)
    & (eligible["avg_position"] <= 20)
    & (eligible["ctr"] < 0.5)
)

print(f"declining_with_demand: {declining_with_demand.sum()} rows ({100*declining_with_demand.mean():.1f}%)")
print(f"low_ctr_visible_page:  {low_ctr_visible_page.sum()} rows ({100*low_ctr_visible_page.mean():.1f}%)")
print(f"median impressions_90d: {eligible['impressions_90d'].median():.0f}")


rows: 30000 | clients: 32
eligible rows: 30000
declining_with_demand: 13152 rows (43.8%)
low_ctr_visible_page:  9759 rows (32.5%)
median impressions_90d: 731


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- An **observed, decision-support ranking**: "these pages show more of the patterns associated with past declines/low-CTR than other pages, based on measured signals."
- A **directional** result: some signals (trend, freshness, CTR gap vs. position tier) are associated with movement in the data I have.
- If my scoring beats the rule baseline on client-holdout validation, that's evidence the pattern is learnable from this data — not proof it's causal.

**What I will never claim:**
- That refreshing a page **will cause** it to recover — that needs an experiment (e.g. a before/after test), not this observational data.
- That I've reverse-engineered a Google ranking factor, or that I'm "predicting Google."
- That the current-window `trend_direction == "down"` label is a real future outcome — the lane guide flags this as a beginner proxy label. For the capstone model in later weeks I'll move to a real prior-window → future-window label instead of leaning on this current-window bucket.
- Any claim beyond what client-holdout validation actually supports — no page-level guarantees, only queue-level, ranked recommendations for a human to review.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.